In [1]:
import os, shutil

def thu_muc(goc, sau, bo_qua=('keyframes',)):
    """Thư mục con tới độ sâu `sau`. Không chui vào `bo_qua` — tầng ảnh."""
    muc, ra = [goc], []
    for _ in range(sau):
        tiep = []
        for d in muc:
            if os.path.basename(d) in bo_qua:
                continue
            try:
                tiep += [e.path for e in os.scandir(d) if e.is_dir()]
            except OSError:
                pass
        ra += tiep
        muc = tiep
    return ra

DIRS = thu_muc('/kaggle/input', 6)
goc = next((d for d in DIRS
            if os.path.exists(os.path.join(d, 'thu_nghiem_dich_vi.py'))), None)
assert goc, "Không thấy code"

shutil.copytree(goc, '/kaggle/working/sys', dirs_exist_ok=True)
os.chdir('/kaggle/working/sys')
print("nguồn:", goc)
print("cache viết lại:", os.path.exists('out/viet_lai_truy_van.json'))

nguồn: /kaggle/input/datasets/ojanek/system-thu-0819
cache viết lại: True


# ***NGHIỆM THU 18/8***

In [2]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['GEMINI_API_KEY'] = UserSecretsClient().get_secret("GEMINI_API_KEY")
print("key:", len(os.environ['GEMINI_API_KEY']), "ký tự")

key: 53 ký tự


In [3]:
import os

def tim(ten):
    return sorted(d for d in DIRS if os.path.basename(d) == ten)

nhe, so400m, base, kf = (tim('aic2026_data_nhe'), tim('aic2026_siglip2_so400m'),
                         tim('aic2026_siglip2'), tim('keyframes'))
for ten, v in [('data_nhe', nhe), ('so400m', so400m), ('base', base)]:
    print(f"{ten:9s}", v[0] if v else "❌ KHÔNG THẤY")
print(f"keyframes {len(kf)} thư mục")
assert nhe and so400m and base, "Thiếu dataset"

cfg = (f"extends: config.example.yaml\n"
       f"offline:\n"
       f"  btc_data_root: {nhe[0]}\n"
       f"  image_feature_root: {so400m[0]}\n"
       f"retrieval:\n  hop_nhat:\n    phu:\n"
       f"    - image_feature_root: {base[0]}\n"
       f"qa:\n  image_roots:\n"
       + "".join(f"  - {p}\n" for p in kf))
open('config_kaggle.yaml', 'w', encoding='utf-8').write(cfg)
print("\n" + cfg[:300])

data_nhe  /kaggle/input/datasets/ojanek/aic2026-data-nhe/aic2026_data_nhe
so400m    /kaggle/input/datasets/ojanek/aic2026-siglip2-so400m/aic2026_siglip2_so400m
base      /kaggle/input/datasets/ojanek/aic2026-siglip2/aic2026_siglip2
keyframes 14 thư mục

extends: config.example.yaml
offline:
  btc_data_root: /kaggle/input/datasets/ojanek/aic2026-data-nhe/aic2026_data_nhe
  image_feature_root: /kaggle/input/datasets/ojanek/aic2026-siglip2-so400m/aic2026_siglip2_so400m
retrieval:
  hop_nhat:
    phu:
    - image_feature_root: /kaggle/input/datasets/oj


In [4]:
!cd /kaggle/working/sys && python -X utf8 thu_nghiem_dich_vi.py \
    --config config_kaggle.yaml --encoder theo-config --hop-nhat \
    --llm --llm-model gemini-3.5-flash-lite --to-hop

  ĐỀ TIẾNG VIỆT — DỊCH MÁY CÓ LẤY LẠI ĐƯỢC ĐIỂM KHÔNG?
  config          : /kaggle/working/sys/config_kaggle.yaml
  kế thừa từ     : /kaggle/working/sys/config.example.yaml
  encoder         : siglip2 google/siglip2-so400m-patch14-384
  vector ảnh      : /kaggle/input/datasets/ojanek/aic2026-siglip2-so400m/aic2026_siglip2_so400m
  đáp án QA       : vi (dịch: opus_mt)
  QA VLM          : gemini_vlm/gemini-3.6-flash -> gemini_vlm/gemini-3.7-flash -> gemini_vlm/gemini-3.5-flash -> gemini_vlm/gemini-3.5-flash-lite -> gemini_vlm/gemini-3.1-flash-lite -> cloudflare_vlm/@cf/meta/llama-3.2-11b-vision-instruct
  vlm_top_n       : 10 · loại câu: ['color', 'open']
  portfolio       : tắt
  ngôn ngữ đề     : en (bản dùng để truy hồi)
  số dòng/truy vấn: 100

Dev set: 120 câu (kis) · top-k 100

Dịch:
  opus-mt: dùng lại 120 bản dịch đã lưu
  LLM viết lại: GeminiVLMProvider/gemini-3.5-flash-lite · 3 bản/câu · lô 10 · hạn 180s
     0 request · 120 cache hit · 0.0s
  bản dịch đã lưu: /kaggle/working/s

In [5]:
!cd /kaggle/working/sys && zip -qr /kaggle/working/out_backup.zip out
!ls -la /kaggle/working/out_backup.zip

-rw-r--r-- 1 root root 81596 Aug 18 23:55 /kaggle/working/out_backup.zip
